# Importación de librerías (SVM)

In [3]:
# ==========================================
# CELDA 1: LIBRERÍAS (CP-09)
# ==========================================
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import ADASYN

df_info = pd.read_csv('./../../dataset/oulad/studentInfo.csv')
df_vle = pd.read_csv('./../../dataset/oulad/studentVle.csv')

df_info['target'] = df_info['final_result'].apply(lambda x: 1 if x == 'Withdrawn' else 0)
presentaciones_2013 = ['2013B', '2013J']
presentaciones_2014 = ['2014B', '2014J']
df_train_info = df_info[df_info['code_presentation'].isin(presentaciones_2013)].copy()
df_test_info = df_info[df_info['code_presentation'].isin(presentaciones_2014)].copy()

# Función de preparación y bucle de entrenamiento

In [6]:
# ==========================================
# CELDA 2: BUCLE DE EXPERIMENTACIÓN AVANZADO (SVM + TRADE-OFF)
# ==========================================
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import ADASYN

# 0. Función de preparación de datos (empaquetada en la celda)
def preparar_datos(df_vle, df_info_split, max_dias):
    vle_f = df_vle[df_vle['date'] <= max_dias].groupby(['id_student', 'code_module', 'code_presentation'])['sum_click'].sum().reset_index()
    df = pd.merge(df_info_split, vle_f.rename(columns={'sum_click': 'total_clicks'}), on=['id_student', 'code_module', 'code_presentation'], how='left').fillna(0)
    return df[['total_clicks']], df['target']

# 1. Wrapper personalizado para SVM
class ThresholdWrapperSVM(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator=None, threshold=0.5):
        self.estimator = estimator
        self.threshold = threshold

    def fit(self, X, y):
        self.estimator.fit(X, y)
        self.classes_ = self.estimator.classes_
        return self

    def predict(self, X):
        proba = self.estimator.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.estimator.predict_proba(X)

# 2. Configuración de métricas
scoring_metrics = {
    'Accuracy': make_scorer(accuracy_score),
    'Precision': make_scorer(precision_score, zero_division=0),
    'Recall': make_scorer(recall_score, zero_division=0),
    'F1-Score': make_scorer(f1_score, zero_division=0)
}

# 3. Espacio de búsqueda reducido para mitigar cuello de botella
param_grid_svm = {
    'clasificador__estimator__C': [0.1, 1.0],
    'clasificador__threshold': [0.3, 0.4, 0.5, 0.6]
}

ventanas_temporales = [30, 60, 90]
estrategias_balanceo = ['Línea Base (Sin balanceo)', 'Undersampling', 'ADASYN']

resultados_mejores = []
todas_las_permutaciones = []

print("Iniciando búsqueda exhaustiva CP-09 (SVM Optimizado)...")

for dias in ventanas_temporales:
    print(f"--- Evaluando ventana a {dias} días ---")

    # Llamada a la función que ahora sí está definida arriba
    X_train, y_train = preparar_datos(df_vle, df_train_info, dias)
    X_test, y_test = preparar_datos(df_vle, df_test_info, dias)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for estrategia in estrategias_balanceo:
        if estrategia == 'Undersampling':
            sampler = RandomUnderSampler(random_state=42)
        elif estrategia == 'ADASYN':
            sampler = ADASYN(random_state=42, n_neighbors=5)
        else:
            sampler = None

        pasos = []
        if sampler is not None:
            pasos.append(('sampler', sampler))

        # Configuración de SVM para soportar probabilidades y caché ampliada
        svm_base = SVC(probability=True, cache_size=2000, random_state=42)
        wrapper = ThresholdWrapperSVM(estimator=svm_base)
        pasos.append(('clasificador', wrapper))

        pipeline = ImbPipeline(pasos)

        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid_svm,
            cv=3,
            scoring=scoring_metrics,
            refit='Recall',
            n_jobs=-1,
            return_train_score=False
        )

        grid_search.fit(X_train_scaled, y_train)

        mejor_modelo = grid_search.best_estimator_
        y_pred = mejor_modelo.predict(X_test_scaled)

        resultados_mejores.append({
            'Días': dias,
            'Estrategia': estrategia,
            'Mejor C': grid_search.best_params_['clasificador__estimator__C'],
            'Mejor Threshold': grid_search.best_params_['clasificador__threshold'],
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall': recall_score(y_test, y_pred, zero_division=0),
            'F1-Score': f1_score(y_test, y_pred, zero_division=0)
        })

        cv_results = grid_search.cv_results_
        for i in range(len(cv_results['params'])):
            todas_las_permutaciones.append({
                'Días': dias,
                'Estrategia': estrategia,
                'C': cv_results['param_clasificador__estimator__C'][i],
                'Threshold': cv_results['param_clasificador__threshold'][i],
                'CV_Accuracy': cv_results['mean_test_Accuracy'][i],
                'CV_Precision': cv_results['mean_test_Precision'][i],
                'CV_Recall': cv_results['mean_test_Recall'][i],
                'CV_F1': cv_results['mean_test_F1-Score'][i]
            })

        print(f"  > {estrategia}: Procesado.")

df_mejores = pd.DataFrame(resultados_mejores)
df_todas_permutaciones = pd.DataFrame(todas_las_permutaciones)

# ==========================================
# PROCESAMIENTO FINAL: AISLAR EL MEJOR ESCENARIO
# ==========================================
mejor_fila = df_mejores.sort_values(by=['Recall', 'F1-Score'], ascending=[False, False]).iloc[0]
mejor_dia = mejor_fila['Días']
mejor_estrategia = mejor_fila['Estrategia']
mejor_c = mejor_fila['Mejor C']

df_tradeoff = df_todas_permutaciones[
    (df_todas_permutaciones['Días'] == mejor_dia) &
    (df_todas_permutaciones['Estrategia'] == mejor_estrategia) &
    (df_todas_permutaciones['C'] == mejor_c)
].copy()

df_tradeoff = df_tradeoff.sort_values(by=['Threshold'])

print("\n=== TABLA 1: MÉTRICAS DEL MEJOR MODELO POR ESCENARIO (TEST) ===")
display(df_mejores)

print(f"\n=== TABLA 2: PROGRESIÓN DE MÉTRICAS - MEJOR ESCENARIO ({mejor_dia} Días | {mejor_estrategia} | C={mejor_c}) ===")
display(df_tradeoff)

Iniciando búsqueda exhaustiva CP-09 (SVM Optimizado)...
--- Evaluando ventana a 30 días ---
  > Línea Base (Sin balanceo): Procesado.
  > Undersampling: Procesado.
  > ADASYN: Procesado.
--- Evaluando ventana a 60 días ---
  > Línea Base (Sin balanceo): Procesado.
  > Undersampling: Procesado.
  > ADASYN: Procesado.
--- Evaluando ventana a 90 días ---
  > Línea Base (Sin balanceo): Procesado.
  > Undersampling: Procesado.
  > ADASYN: Procesado.

=== TABLA 1: MÉTRICAS DEL MEJOR MODELO POR ESCENARIO (TEST) ===


,Días,Estrategia,Mejor C,Mejor Threshold,Accuracy,Precision,Recall,F1-Score
0,30,Línea Base (Sin balanceo),1.0,0.3,0.712967,0.584632,0.518714,0.549704
1,30,Undersampling,1.0,0.3,0.338019,0.337829,0.999845,0.505020
2,30,ADASYN,0.1,0.3,0.337757,0.337757,1.000000,0.504960
3,60,Línea Base (Sin balanceo),0.1,0.3,0.720153,0.587508,0.575555,0.581470
4,60,Undersampling,1.0,0.3,0.337757,0.337757,1.000000,0.504960
5,60,ADASYN,0.1,0.3,0.337757,0.337757,1.000000,0.504960
6,90,Línea Base (Sin balanceo),0.1,0.3,0.733949,0.605756,0.608014,0.606883
7,90,Undersampling,1.0,0.3,0.476553,0.382812,0.897966,0.536787
8,90,ADASYN,1.0,0.3,0.337757,0.337757,1.000000,0.504960



=== TABLA 2: PROGRESIÓN DE MÉTRICAS - MEJOR ESCENARIO (30 Días | ADASYN | C=0.1) ===


,Días,Estrategia,C,Threshold,CV_Accuracy,CV_Precision,CV_Recall,CV_F1
16,30,ADASYN,0.1,0.3,0.274740,0.274740,1.000000,0.430239
17,30,ADASYN,0.1,0.4,0.537517,0.359505,0.757605,0.477178
18,30,ADASYN,0.1,0.5,0.603521,0.409261,0.666452,0.484410
19,30,ADASYN,0.1,0.6,0.712910,0.510496,0.513434,0.492066
